
# Laboratorio 3 (adaptado al español): Automatización de soporte al cliente con CrewAI

Este notebook adapta el laboratorio original **L3_customer_support.ipynb** a un ejemplo completamente en español y listo para ejecutarse en **Google Colab**.

## Objetivos de aprendizaje
En este laboratorio vas a practicar los **seis elementos clave** que mejoran el desempeño de los agentes:

1. **Role Playing (roles)**
2. **Focus (enfoque)**
3. **Tools (herramientas)**
4. **Cooperation (cooperación)**
5. **Guardrails (barandas de seguridad)**
6. **Memory (memoria)**

---

## Caso de estudio
Vamos a construir un sistema multiagente para responder solicitudes de soporte de clientes en español.

El sistema tendrá dos agentes:

- **Representante Senior de Soporte**
- **Especialista de Aseguramiento de Calidad**

El primer agente redacta la respuesta y el segundo la revisa antes de entregarla al cliente.



## 1. Instalación de librerías

Si estás en Colab, ejecuta esta celda para instalar las dependencias.


In [ ]:

!pip -q install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29



## 2. Configuración inicial

Aquí configuramos:
- control de warnings
- credenciales
- modelo base


In [ ]:
import os
import warnings
from getpass import getpass

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = 'sk-xxx'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = 'xxx'
# Puedes cambiar el modelo si deseas
#os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

print("Modelo configurado:", os.environ["OPENAI_MODEL_NAME"])
print("SERPER_API_KEY cargada:", "Sí" if os.environ.get("SERPER_API_KEY") else "No")



## 3. Importaciones principales


In [ ]:

from crewai import Agent, Task, Crew
from IPython.display import Markdown, display



## 4. Role Playing, Focus y Cooperation

Vamos a definir los agentes.

### Idea pedagógica
- **Role Playing**: cada agente recibe un rol concreto.
- **Focus**: cada agente se especializa en su función.
- **Cooperation**: el agente de calidad puede revisar y mejorar la salida del agente de soporte.


### 4.1 Agente: Representante Senior de Soporte

In [ ]:

agente_soporte = Agent(
    role="Representante Senior de Soporte",
    goal="Ofrecer la respuesta más clara, útil, precisa y cordial al cliente",
    backstory=(
        "Trabajas en una empresa que desarrolla sistemas multiagente con CrewAI. "
        "Tu responsabilidad es atender solicitudes de clientes estratégicos de manera profesional. "
        "Debes responder en español, con tono amable, técnico cuando sea necesario, "
        "y asegurarte de no hacer suposiciones injustificadas."
    ),
    allow_delegation=True,
    verbose=True
)



**Observación:**  
Al dejar `allow_delegation=True`, este agente **puede delegar** parte del trabajo si el flujo lo requiere.


### 4.2 Agente: Especialista de Aseguramiento de Calidad

In [ ]:

agente_calidad = Agent(
    role="Especialista de Aseguramiento de Calidad de Soporte",
    goal="Garantizar que la respuesta final al cliente sea correcta, completa y de alta calidad",
    backstory=(
        "Eres responsable de revisar las respuestas del equipo de soporte antes de que lleguen al cliente. "
        "Tu misión es verificar exactitud, claridad, cobertura completa del problema, "
        "tono profesional y utilidad práctica. "
        "Debes detectar omisiones, ambigüedades y mejorar la calidad final de la respuesta."
    ),
    allow_delegation=True,
    verbose=True
)



## 5. Tools, Guardrails y Memory

### Tools
Ahora agregaremos herramientas reales para que los agentes puedan consultar información externa.

Usaremos herramientas de CrewAI para:
- buscar en la web
- analizar páginas específicas
- consultar documentación


In [ ]:

from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool



## 6. Instanciación de herramientas

En este ejemplo usamos:
- `SerperDevTool` para búsqueda web
- `ScrapeWebsiteTool` para leer una URL específica
- `WebsiteSearchTool` para buscar dentro de un sitio/documentación


In [ ]:

herramienta_busqueda = SerperDevTool()
herramienta_scraping = ScrapeWebsiteTool()

herramienta_busqueda_docs = WebsiteSearchTool(
    website="https://docs.crewai.com/"
)



### 6.1 Tool opcional de documentación específica
También podemos fijar una URL concreta de la documentación para que el agente la consulte.


In [ ]:

herramienta_doc_especifica = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)



## 7. Creación de tareas

Vamos a definir dos tareas:

1. **Resolución de la consulta**
2. **Revisión de calidad**

En este caso, las herramientas se pasan a nivel de tarea, no a nivel de agente.


### 7.1 Tarea: Resolver la consulta del cliente

In [ ]:

tarea_resolucion = Task(
    description=(
        "{cliente} ha enviado una consulta importante:\n"
        "{consulta}\n\n"
        "{persona} de {cliente} es quien realizó la solicitud. "
        "Debes usar toda la información disponible para construir la mejor respuesta posible. "
        "La respuesta debe ser completa, precisa, útil y escrita en español. "
        "Cuando sea pertinente, usa fuentes o referencias de la documentación."
    ),
    expected_output=(
        "Una respuesta detallada y bien estructurada en español para la consulta del cliente, "
        "con explicaciones claras, pasos sugeridos y referencias cuando apliquen."
    ),
    agent=agente_soporte,
    tools=[herramienta_busqueda, herramienta_busqueda_docs, herramienta_doc_especifica]
)


### 7.2 Tarea: Revisión de aseguramiento de calidad

In [ ]:

tarea_revision = Task(
    description=(
        "Revisa la respuesta redactada por el Representante Senior de Soporte para la consulta de {cliente}. "
        "Verifica que la respuesta sea:\n"
        "- completa\n"
        "- correcta\n"
        "- útil\n"
        "- cordial\n"
        "- profesional\n\n"
        "Asegúrate de que todas las partes de la consulta hayan sido atendidas. "
        "Si identificas problemas, mejora la respuesta final antes de entregarla."
    ),
    expected_output=(
        "Una versión final optimizada de la respuesta al cliente, en español, "
        "con calidad profesional de soporte técnico."
    ),
    agent=agente_calidad
)



## 8. Creación del crew

### Memory
Al configurar `memory=True`, habilitamos memoria en el crew.

Esto permite que el sistema:
- mantenga contexto entre tareas,
- aproveche información previa de la ejecución,
- y produzca respuestas más consistentes.


In [ ]:

equipo_soporte = Crew(
    agents=[agente_soporte, agente_calidad],
    tasks=[tarea_resolucion, tarea_revision],
    verbose=2,
    memory=True
)



## 9. Ejecución del crew

Vamos a simular una consulta de soporte en español.


In [ ]:

inputs = {
    "cliente": "Universidad EAFIT",
    "persona": "Profesor Edwin Montoya",
    "consulta": (
        "Necesito ayuda para crear un Crew en CrewAI y ejecutarlo correctamente. "
        "Además, quiero saber cómo activar memoria en el crew y qué recomendaciones hay "
        "para diseñar agentes con roles bien definidos."
    )
}

resultado = equipo_soporte.kickoff(inputs=inputs)


## 10. Visualización del resultado

In [ ]:

display(Markdown(resultado))



## 11. Guardrails: reflexión

Este ejemplo ilustra la idea de **guardrails** de forma práctica:

- los agentes están orientados a permanecer dentro del dominio del soporte,
- las tareas acotan el alcance,
- y el agente de calidad actúa como una segunda capa de validación.

> En sistemas más complejos, los guardrails también pueden incluir:
- límites de iteraciones,
- validación estructurada de salidas,
- control del uso de herramientas,
- filtros de seguridad.



## 12. Probar con otra consulta

Puedes cambiar la consulta por otros casos como:

- configuración de tools
- diferencia entre tareas y agentes
- diseño de workflows jerárquicos
- integración con LangChain


In [ ]:

inputs_2 = {
    "cliente": "Laboratorio de IA Aplicada",
    "persona": "Ana Pérez",
    "consulta": (
        "Quiero entender la diferencia entre asignar tools a nivel de agente y a nivel de tarea en CrewAI. "
        "También necesito recomendaciones para decidir cuándo usar uno u otro enfoque."
    )
}

resultado_2 = equipo_soporte.kickoff(inputs=inputs_2)
display(Markdown(resultado_2))



## 13. Extensiones sugeridas para estudiantes

### Extensión 1
Agregar un tercer agente:
- **Especialista en documentación técnica**

### Extensión 2
Agregar una base local de conocimiento y convertir este laboratorio en un sistema:
- **Soporte + RAG**

### Extensión 3
Cambiar el proceso:
- secuencial
- jerárquico
- paralelo

### Extensión 4
Pedir además una salida estructurada en:
- Markdown
- JSON
- respuesta breve ejecutiva



## 14. Reflexión final

Este laboratorio muestra cómo combinar en un caso de soporte real:

- **roles especializados**
- **tareas bien definidas**
- **herramientas externas**
- **memoria**
- **revisión de calidad**
- **guardrails implícitos**

### Mensaje clave
Un sistema multiagente de soporte no solo responde:
**investiga, estructura, justifica, revisa y mejora la respuesta final**.
